# SCFT Computation with Cylindrical Phase

<!-- This example demonstrates how to find the saddle-point solution of AB diblock copolymer systems with Anderson Mixing. -->
* Fields are efficiently updated using Anderson mixing.

* This example computes the stress residuals to determine the box size that minimizes the free energy: 
\begin{align}
{\bf R}_{stress} = − \frac{1}{Q}\frac{\partial Q}{\partial \boldsymbol\theta}
\end{align}
where $\boldsymbol\theta = [L_x, L_y, L_z]^{T}$ is a vector of unit-cell parameters.

* This example is based on the equations in `PolymerFieldTheory/IncompressibleModel.ipynb` and uses the `symmetric_formulation.py` and `scft.py` modules.

References:
* [(2009) Fast and accurate SCFT calculations for periodic block-copolymer morphologies using the spectral method with Anderson mixing](http://dx.doi.org/10.1140/epje/i2009-10534-3)
* [(2011) Efficiency of pseudo-spectral algorithms with Anderson mixing for the SCFT of periodic block-copolymer phases](http://dx.doi.org/10.1140/epje/i2011-11110-0)
* [(2017) Accelerating self-consistent field theory of block polymers in a variable unit cell](http://dx.doi.org/10.1063/1.4986643)

### 1. Setting simulation parameters

| Parameter | Symbol | Value |
|:----------|:------:|------:|
| Reference chain length | $N_\text{Ref}$ | 90 |
| Contour step | $\Delta s$ | 1/90 |
| Reference length | $R_0$ | $b N_\text{Ref}^{1/2}$ |
| Box size X | $L_x$ | 2.4 $R_0$ |
| Box size Y | $L_y$ | 1.8 $R_0$ |
| Grid points X | $m_x$ | 48 |
| Grid points Y | $m_y$ | 32 |
| Segment length A | $b_A/b$ | 1.0 |
| Segment length B | $b_B/b$ | 1.0 |
| Interaction parameter | $\chi N$ | 15 |
| A-block fraction | $f$ | 1/3 |

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"      # Single-threaded OpenMP

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from polymerfts import scft

# Major Simulation params
f = 1.0/3.0     # A-fraction of major BCP chain, f

params = {
    # "platform":"cpu-mkl",           # choose platform among [cuda, cpu-mkl, cpu-fftw]
    
    "nx":[48,32],                        # Simulation grid numbers
    "lx":[2.4,1.8],  # Simulation box size as a_Ref * N_Ref^(1/2) unit,
                                            # where "a_Ref" is reference statistical segment length
                                            # and "N_Ref" is the number of segments of reference linear homopolymer chain.

    "reduce_memory":False,     # Reduce memory usage by storing only check points.
    "relax_cell":True,     # Find box size that minimizes the free energy during saddle point iteration.
    "chain_model":"discrete",   # "discrete" or "continuous" chain model
    "ds":1/90,                  # Contour step interval, which is equal to 1/N_Ref.

    "segment_lengths":{         # Relative statistical segment length compared to "a_Ref.
        "A":1.0, 
        "B":1.0, },

    "chi_n": {"A,B": 15},       # Interaction parameter, Flory-Huggins params * N_Ref

    "polymers":[{      # Polymer species
        "volume_fraction":1.0,  # volume fraction of polymer chain
        "blocks":[              # AB diBlock Copolymer
            {"type":"A", "length":f, }, # A-block
            {"type":"B", "length":1-f}, # B-block
        ],},],

    "optimizer":{
        "name":"am",            # Anderson Mixing
        "max_hist":20,          # Maximum number of history
        "start_error":1e-2,     # When switch to AM from simple mixing
        "mix_min":0.1,          # Minimum mixing rate of simple mixing
        "mix_init":0.1,         # Initial mixing rate of simple mixing
    },

    "max_iter":2000,     # The maximum relaxation iterations
    "tolerance":1e-8     # Terminate iteration if the self-consistency error is less than tolerance
}

### 2. Initial potential fields

In [ ]:
nx = params["nx"]
lx = params["lx"]

# Set initial fields
w_A = np.zeros(list(params["nx"]), dtype=np.float64)
w_B = np.zeros(list(params["nx"]), dtype=np.float64)
print("w_A and w_B are initialized to cylindrical phase.")
cylinder_positions = [
[0.0,0.0],[1/2,1/2],]
for x,y in cylinder_positions:
    mx, my = np.round((np.array([x, y])*params["nx"])).astype(np.int32)
    w_A[mx,my] = -5/(np.prod(params["lx"])/np.prod(params["nx"]))
w_A = gaussian_filter(w_A, sigma=np.min(params["nx"])/5, mode='wrap')

w = {"A": w_A, "B": w_B}  # potential field dictionary

# Plot each image with the same vmin and vmax
vmin = min(w_A.min(), w_B.min())
vmax = max(w_A.max(), w_B.max())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle("Initial Potential Fields")
im = axes[0].imshow(w_A[:,:], extent=(0, lx[1], 0, lx[0]), origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
im = axes[1].imshow(w_B[:,:], extent=(0, lx[1], 0, lx[0]), origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
axes[0].set(title='$w_A$', xlabel='y', ylabel='x')
axes[1].set(title='$w_B$', xlabel='y', ylabel='x')

fig.subplots_adjust(right=0.92)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
fig.show()

### 3. Initialization

In [ ]:
# Initialize calculation
calculation = scft.SCFT(params=params)

### 4. Run

In [ ]:
# Run
calculation.run(initial_fields={"A": w_A, "B": w_B})

### 5. Display the results

In [ ]:
# Get the box size
lx = calculation.cb.get_lx()
print("Box size:", lx)

# Get the ensemble average concentration for each monomer type
phi_A = np.reshape(calculation.phi["A"], nx)
phi_B = np.reshape(calculation.phi["B"], nx)

# Plot each image with the same vmin and vmax
vmin = min(phi_A.min(), phi_B.min())
vmax = max(phi_A.max(), phi_B.max())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle("Concentrations")
im = axes[0].imshow(phi_A, extent=(0, lx[1], 0, lx[0]), origin='lower', cmap='RdBu_r', vmin=vmin, vmax=vmax)
im = axes[1].imshow(phi_B, extent=(0, lx[1], 0, lx[0]), origin='lower', cmap='RdBu_r', vmin=vmin, vmax=vmax)
axes[0].set(title=r'$\phi_A$', xlabel='y', ylabel='x')
axes[1].set(title=r'$\phi_B$', xlabel='y', ylabel='x')

fig.subplots_adjust(right=0.92)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
fig.show()

### 6. Save the results

In [ ]:
# Save final results (.mat, .json or .yaml format)
calculation.save_results("fields.json")

# Recording first a few iteration results for debugging and refactoring
    #        1   -9.598E-16  [ 1.1360086E+00  ]     2.719622865   1.6603908E+00  lx=[  2.4000000, 1.8000000 ]
    #        2   -1.364E-16  [ 1.0855029E+00  ]     2.510555386   1.2228642E+00  lx=[  2.4019552, 1.8023108 ]
    #        3    2.210E-16  [ 1.0635675E+00  ]     2.418668516   8.7842601E-01  lx=[  2.4028003, 1.8031295 ]
    #        4   -6.138E-16  [ 1.0536527E+00  ]     2.375149769   6.1799778E-01  lx=[  2.4031693, 1.8033201 ]
    #        5   -6.294E-17  [ 1.0492962E+00  ]     2.353270078   4.2969988E-01  lx=[  2.4033221, 1.8032349 ]
